# Optimización de Hiperparámetros

Notebook para búsqueda sistemática de hiperparámetros con visualización en tiempo real.

**Modelos**: XGBoost, Random Forest, LSTM  
**Dataset**: 56,072 muestras · Split temporal 70/15/15  
**Tiempo estimado**: ~45-60 min total

In [ ]:
%matplotlib inline

import json
import sys
import time
import itertools
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# Proyecto en path
sys.path.insert(0, str(Path.cwd().parent))
from backtest.ml_models import (
    _load_dataset, _samples_to_xy, _temporal_split, extract_features,
)

# Estilo de gráficas
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "figure.facecolor": "white"})

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
CONFIGS_DIR = Path("configs")
DATASET_PATH = Path("../backtest/data/labeled/dataset.jsonl")

print("✅ Imports listos")


## Carga de Datos y Split Temporal

In [ ]:
%%time

samples = _load_dataset(str(DATASET_PATH))
train_s, val_s, test_s = _temporal_split(samples, 0.70, 0.15)

X_train, y_train = _samples_to_xy(train_s)
X_val, y_val = _samples_to_xy(val_s)
X_test, y_test = _samples_to_xy(test_s)

# Train+Val combinado para CV
X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

print(f"Total: {len(samples):,}")
print(f"Train: {len(train_s):,} | Val: {len(val_s):,} | Test: {len(test_s):,}")
print(f"Features: {X_train.shape[1]}")
print(f"Clase LONG: {y_train.mean():.1%} | SHORT: {1 - y_train.mean():.1%}")

# Distribución de clases
fig, ax = plt.subplots(figsize=(5, 3))
counts = pd.Series(y_train).value_counts().sort_index()
ax.bar(["SHORT (0)", "LONG (1)"], counts.values, color=["#e74c3c", "#2ecc71"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")
ax.set_ylabel("Muestras")
ax.set_title("Distribución de Clases (Train)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png")
plt.show()


## 1. XGBoost — Grid Search

Grid reducido para tiempo razonable (~15 min). Usa `TimeSeriesSplit` para respetar la temporalidad.

In [ ]:
%%time
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Grid reducido (81 combinaciones × 3 folds = 243 fits)
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
}

base = xgb.XGBClassifier(
    eval_metric="logloss", random_state=42,
)
cv = TimeSeriesSplit(n_splits=3)

search_xgb = GridSearchCV(
    base, param_grid, cv=cv, scoring="accuracy",
    verbose=1, n_jobs=-1, return_train_score=True,
)
search_xgb.fit(X_cv, y_cv)

print(f"\n🏆 Mejor accuracy: {search_xgb.best_score_:.4f}")
print(f"   Params: {search_xgb.best_params_}")


In [ ]:
# Heatmap: max_depth vs learning_rate
results_df = pd.DataFrame(search_xgb.cv_results_)

# Pivot para heatmap (promedio sobre n_estimators)
pivot_data = results_df.copy()
for col in ["param_max_depth", "param_learning_rate", "param_n_estimators"]:
    pivot_data[col] = pivot_data[col].astype(float)

heatmap_df = pivot_data.groupby(
    ["param_max_depth", "param_learning_rate"]
)["mean_test_score"].max().unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGn", ax=axes[0])
axes[0].set_title("XGBoost: max_depth vs learning_rate (mejor score)")
axes[0].set_xlabel("learning_rate")
axes[0].set_ylabel("max_depth")

# Top 10 configuraciones
top10 = results_df.nsmallest(10, "rank_test_score")
labels = [
    f"d={r['param_max_depth']}, lr={r['param_learning_rate']}, n={r['param_n_estimators']}"
    for _, r in top10.iterrows()
]
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(top10))]
axes[1].barh(labels[::-1], top10["mean_test_score"].values[::-1], color=colors[::-1])
axes[1].set_xlabel("Accuracy (CV)")
axes[1].set_title("XGBoost: Top 10 Configuraciones")
axes[1].axvline(search_xgb.best_score_, color="red", ls="--", alpha=0.5)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "xgboost_optimization.png")
plt.show()

# Guardar resultados
xgb_results = {
    "model": "xgboost",
    "best_params": search_xgb.best_params_,
    "best_score": float(search_xgb.best_score_),
    "all_results": [
        {"params": dict(r["params"]), "mean_score": float(r["mean_test_score"]),
         "std_score": float(r["std_test_score"]), "rank": int(r["rank_test_score"])}
        for _, r in results_df.iterrows()
    ][:30],
    "timestamp": datetime.now().isoformat(),
}
(RESULTS_DIR / "xgboost_optimization.json").write_text(json.dumps(xgb_results, indent=2))
print(f"✅ Resultados guardados en {RESULTS_DIR / 'xgboost_optimization.json'}")


## 2. Random Forest — Random Search

50 iteraciones aleatorias con `TimeSeriesSplit`.

In [ ]:
%%time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [6, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5, 10],
    "max_features": ["sqrt", "log2"],
}

base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
cv = TimeSeriesSplit(n_splits=3)

search_rf = RandomizedSearchCV(
    base_rf, param_dist, n_iter=50, cv=cv, scoring="accuracy",
    verbose=1, n_jobs=-1, random_state=42, return_train_score=True,
)
search_rf.fit(X_cv, y_cv)

print(f"\n🏆 Mejor accuracy: {search_rf.best_score_:.4f}")
print(f"   Params: {search_rf.best_params_}")


In [ ]:
# Feature importance del mejor modelo
best_rf = search_rf.best_estimator_

# Nombres de features
TF = ["1h", "4h", "1d"]
KEYS = ["price", "rsi", "macd_hist", "adx", "volume_ratio", "atr_ratio", "bb_pos", "heatmap", "structure"]
feat_names = [f"{tf}_{k}" for tf in TF for k in KEYS] + ["tf_agreement", "rsi_divergence", "vol_spread"]
assert len(feat_names) == X_train.shape[1], f"Feature names ({len(feat_names)}) != features ({X_train.shape[1]})"

importances = best_rf.feature_importances_
idx = np.argsort(importances)[-15:]  # Top 15

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([feat_names[i] for i in idx], importances[idx], color="#3498db")
ax.set_xlabel("Importancia")
ax.set_title("Random Forest: Top 15 Features")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rf_feature_importance.png")
plt.show()

# Guardar resultados
rf_df = pd.DataFrame(search_rf.cv_results_)
rf_results = {
    "model": "random_forest",
    "best_params": {k: (None if v is None else v) for k, v in search_rf.best_params_.items()},
    "best_score": float(search_rf.best_score_),
    "all_results": [
        {"params": {k: (None if v is None else v) for k, v in r["params"].items()},
         "mean_score": float(r["mean_test_score"]),
         "std_score": float(r["std_test_score"]), "rank": int(r["rank_test_score"])}
        for _, r in rf_df.iterrows()
    ][:30],
    "timestamp": datetime.now().isoformat(),
}
(RESULTS_DIR / "random_forest_optimization.json").write_text(json.dumps(rf_results, indent=2, default=str))
print(f"✅ Resultados guardados en {RESULTS_DIR / 'random_forest_optimization.json'}")


## 3. LSTM — Búsqueda Manual

10 configuraciones aleatorias con early stopping. Usa MPS/CUDA si está disponible.

In [ ]:
%%time
import torch
import torch.nn as nn

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {device}")

# Normalizar features
X_all = np.array([extract_features(s["indicators"]) for s in samples], dtype=np.float32)
y_all = np.array([1 if s["label"]["bias"] == "LONG" else 0 for s in samples], dtype=np.int32)
n_train, n_val = len(train_s), len(val_s)

mean = X_all[:n_train].mean(axis=0)
std = X_all[:n_train].std(axis=0) + 1e-8
X_norm = (X_all - mean) / std
input_size = X_norm.shape[1]

# Grid reducido: 10 configs
lstm_grid = {
    "hidden_size": [32, 64, 128],
    "num_layers": [1, 2],
    "sequence_length": [5, 10],
    "learning_rate": [0.001, 0.005],
    "dropout": [0.1, 0.2],
    "batch_size": [32],
}
all_combos = list(itertools.product(*lstm_grid.values()))
random.seed(42)
combos = random.sample(all_combos, min(10, len(all_combos)))
keys = list(lstm_grid.keys())

def build_sequences(features, labels, seq_len):
    n, d = features.shape
    seqs = np.zeros((n, seq_len, d), dtype=np.float32)
    for i in range(n):
        start = max(0, i - seq_len + 1)
        s = features[start:i + 1]
        seqs[i, seq_len - len(s):] = s
    return torch.tensor(seqs, device=device), torch.tensor(labels, dtype=torch.long, device=device)

lstm_results = []
train_histories = {}

for ci, combo in enumerate(tqdm(combos, desc="LSTM configs")):
    params = dict(zip(keys, combo))
    sl = params["sequence_length"]
    hs = params["hidden_size"]
    nl = params["num_layers"]
    lr = params["learning_rate"]
    dr = params["dropout"]
    bs = params["batch_size"]

    X_tr, y_tr = build_sequences(X_norm[:n_train], y_all[:n_train], sl)
    X_va, y_va = build_sequences(X_norm[:n_train + n_val], y_all[:n_train + n_val], sl)
    X_va, y_va = X_va[n_train:], y_va[n_train:]

    class LSTMNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hs, nl, dropout=dr if nl > 1 else 0, batch_first=True)
            self.fc = nn.Linear(hs, 2)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])

    model = LSTMNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_vl, wait, best_state = float("inf"), 0, None
    t_losses, v_losses = [], []

    for epoch in range(50):
        model.train()
        perm = torch.randperm(len(X_tr), device=device)
        e_loss = 0.0
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = criterion(model(X_tr[idx]), y_tr[idx])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            e_loss += loss.item() * len(idx)
        t_losses.append(e_loss / len(X_tr))

        model.eval()
        with torch.no_grad():
            vl = criterion(model(X_va), y_va).item()
        v_losses.append(vl)

        if vl < best_vl:
            best_vl = vl; wait = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 10: break

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        acc = float((model(X_va).argmax(1).cpu().numpy() == y_va.cpu().numpy()).mean())

    lstm_results.append({"params": params, "mean_score": acc, "epochs": epoch + 1})
    train_histories[ci] = {"train": t_losses, "val": v_losses, "acc": acc, "params": params}
    print(f"  Config {ci+1}: acc={acc:.4f} epochs={epoch+1} h={hs} l={nl} sl={sl} lr={lr}")
    del model, optimizer, X_tr, y_tr, X_va, y_va
    if device.type == "mps": torch.mps.empty_cache()

lstm_results.sort(key=lambda r: r["mean_score"], reverse=True)
print(f"\n🏆 Mejor LSTM: {lstm_results[0]['mean_score']:.4f}")
print(f"   Params: {lstm_results[0]['params']}")


In [ ]:
# Curvas de loss para top 3 configuraciones
top3_idx = sorted(train_histories.keys(), key=lambda k: train_histories[k]["acc"], reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, idx in zip(axes, top3_idx):
    h = train_histories[idx]
    epochs = range(1, len(h["train"]) + 1)
    ax.plot(epochs, h["train"], label="Train", color="#3498db")
    ax.plot(epochs, h["val"], label="Val", color="#e74c3c")
    ax.set_title(f"Config {idx+1} · acc={h['acc']:.4f}\nh={h['params']['hidden_size']} l={h['params']['num_layers']}")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Loss")
plt.suptitle("LSTM: Curvas de Entrenamiento (Top 3)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lstm_training_curves.png", bbox_inches="tight")
plt.show()

# Guardar resultados
lstm_out = {
    "model": "lstm",
    "best_params": lstm_results[0]["params"],
    "best_score": lstm_results[0]["mean_score"],
    "all_results": lstm_results,
    "timestamp": datetime.now().isoformat(),
}
(RESULTS_DIR / "lstm_optimization.json").write_text(json.dumps(lstm_out, indent=2))
print(f"✅ Resultados guardados en {RESULTS_DIR / 'lstm_optimization.json'}")


## 4. Comparación de Modelos

In [ ]:
# Cargar todos los resultados
model_scores = {}
model_params = {}
for name in ["xgboost", "random_forest", "lstm"]:
    path = RESULTS_DIR / f"{name}_optimization.json"
    if path.exists():
        data = json.loads(path.read_text())
        model_scores[name] = data["best_score"]
        model_params[name] = data["best_params"]

# Gráfica comparativa
fig, ax = plt.subplots(figsize=(8, 4))
names = list(model_scores.keys())
scores = [model_scores[n] for n in names]
colors = ["#2ecc71", "#3498db", "#9b59b6"][:len(names)]
bars = ax.bar(names, scores, color=colors, width=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{score:.4f}", ha="center", fontweight="bold")
ax.set_ylabel("Accuracy (CV)")
ax.set_title("Comparación: Mejor Accuracy por Modelo")
ax.set_ylim(min(scores) - 0.02, max(scores) + 0.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png")
plt.show()

# Tabla resumen
print("\n" + "=" * 70)
print(f"{'Modelo':<18} {'Accuracy':>10}   Mejores Hiperparámetros")
print("-" * 70)
for name in names:
    params_str = ", ".join(f"{k}={v}" for k, v in model_params[name].items())
    print(f"{name:<18} {model_scores[name]:>10.4f}   {params_str}")
print("=" * 70)

winner = max(model_scores, key=model_scores.get)
print(f"\n🏆 Recomendación: {winner} con accuracy {model_scores[winner]:.4f}")


## 5. Aplicar Mejores Hiperparámetros

Genera el código para actualizar `ml_models.py` con los mejores parámetros encontrados.

In [ ]:
# Resumen consolidado
summary = {"timestamp": datetime.now().isoformat(), "models": {}}

for name in ["xgboost", "random_forest", "lstm"]:
    path = RESULTS_DIR / f"{name}_optimization.json"
    if path.exists():
        data = json.loads(path.read_text())
        summary["models"][name] = {
            "best_params": data["best_params"],
            "best_score": data["best_score"],
        }

(RESULTS_DIR / "best_params_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("✅ Resumen guardado en results/best_params_summary.json\n")

# Código sugerido para ml_models.py
if "xgboost" in summary["models"]:
    p = summary["models"]["xgboost"]["best_params"]
    print("# XGBoostPredictor.train() — actualizar constructor:")
    print(f"self.model = xgb.XGBClassifier(")
    for k, v in p.items():
        print(f"    {k}={repr(v)},")
    print(f"    eval_metric='logloss', random_state=42,")
    print(f")\n")

if "random_forest" in summary["models"]:
    p = summary["models"]["random_forest"]["best_params"]
    print("# RandomForestPredictor.train() — actualizar constructor:")
    print(f"self.model = RandomForestClassifier(")
    for k, v in p.items():
        print(f"    {k}={repr(v)},")
    print(f"    random_state=42, n_jobs=-1,")
    print(f")\n")

if "lstm" in summary["models"]:
    p = summary["models"]["lstm"]["best_params"]
    print("# LSTMPredictor.__init__() — actualizar defaults:")
    for k, v in p.items():
        print(f"#   {k} = {repr(v)}")


## 6. Siguientes Pasos

1. **Aplicar parámetros**: Actualizar `ml_models.py` con los mejores hiperparámetros
2. **Re-entrenar**: Ejecutar `backtest/train_models.py` con los nuevos parámetros
3. **Evaluar en test set**: Comparar accuracy en el split de test (15% final)
4. **Backtest completo**: Correr el backtest con los modelos optimizados
5. **Fine-tuning LSTM**: Si LSTM ganó, explorar grid más fino alrededor del mejor punto
6. **Ensemble**: Considerar combinar predicciones de los 3 modelos